# Bài tập nhóm NLP - Kỹ sư Prompting GPT

Notebook này được dùng để thực hiện từng bước các nhiệm vụ:
1. Đọc dữ liệu (Lọc ra 37 text cần summarize và 11 groundtruth)
2. Thiết kế Few-shot Prompt
3. Gọi API GPT-4o-mini và đo lường (chạy 37 mẫu thực tế)
4. Tính toán các chỉ số tự động cho TỪNG ID (37 rows)

In [17]:
import pandas as pd
import json
import time

# Đọc dữ liệu Pool
df_pool = pd.read_csv('NLP.FSB Pool - Pool.csv')

# Xem các cột để chắc chắn
print("Các cột trong file:", df_pool.columns.tolist())
display(df_pool.head(2))

Các cột trong file: ['Mô tả thô', 'ID', 'Tiêu đề chuẩn', 'Mô tả chuẩn']


,Mô tả thô,ID,Tiêu đề chuẩn,Mô tả chuẩn
0,Nơ Trang Long 77.7/112.8 5 5.5 26 27.5 tỷ Bình...,SYS-20264727-938,NaN,Mặt tiền Nơ Trang Long Bình Thạnh - 77.7m2 (5....
1,Lê Văn Sỹ 46 4 4.6 10 12.8 tỷ Nhiêu Lộc Quận 3...,SYS-MP75438B-O8,NaN,Hẻm xe hơi Lê Văn Sỹ Quận 3 - 46m2 (4.6x10) - ...


### Bước 2: Lọc dữ liệu Text và Groundtruth
Dữ liệu hiện tại bao gồm:
- **Cột A (Mô tả thô)**: Chứa 37 đoạn văn bản (text) cần được summarize.
- **Cột B (ID)**: Chứa mã định danh.
- **Cột D (Mô tả chuẩn)**: Chứa 11 đoạn tóm tắt mẫu (groundtruth).

Chúng ta sẽ tiến hành trích xuất chúng ra để phục vụ cho việc test và thiết kế Prompt.

In [18]:
col_text = df_pool.columns[0]  # Cột A: Mô tả thô
col_id = df_pool.columns[1]    # Cột B: ID
col_gt = df_pool.columns[3]    # Cột D: Mô tả chuẩn

# 1. Lọc các dòng có text cần summarize (Cột A không bị rỗng)
df_texts = df_pool[df_pool[col_text].notna()]
print(f"Số lượng text tìm thấy ở cột A: {len(df_texts)}")

# 2. Lọc các dòng có groundtruth (Cột D không bị rỗng)
df_groundtruth = df_pool[df_pool[col_gt].notna()]
print(f"Số lượng groundtruth tìm thấy ở cột D: {len(df_groundtruth)}")

# Lấy 2 mẫu đầu tiên làm few-shot examples
few_shot_examples = df_groundtruth[[col_text, col_gt]].to_dict('records')[:2]

# Chuyển toàn bộ 37 mẫu thành test_data
test_data = df_texts.to_dict('records')

print("\nĐã trích xuất xong dữ liệu!")

Số lượng text tìm thấy ở cột A: 37
Số lượng groundtruth tìm thấy ở cột D: 11

Đã trích xuất xong dữ liệu!


### Bước 3: Thiết kế Few-shot Prompt và Gọi API GPT-4o-mini cho 37 ID

Lần này chúng ta sẽ chạy vòng lặp qua toàn bộ 37 mẫu Text để lấy prediction đầy đủ.

In [ ]:
# !pip install openai
from openai import OpenAI
import time
import json

# THAY API KEY CỦA BẠN VÀO ĐÂY
OPENAI_API_KEY = 'your_openai_api_key_here'
client = OpenAI(api_key=OPENAI_API_KEY)

# System Prompt giải thích nhiệm vụ
system_prompt = """
Bạn là một trợ lý AI chuyên nghiệp về Bất động sản.
Nhiệm vụ của bạn là đọc 'Mô tả thô' của một căn nhà và tóm tắt lại thành một bài đăng quảng cáo 'Mô tả chuẩn' với cấu trúc rõ ràng, chuyên nghiệp.
Hãy giữ nguyên các thông tin quan trọng như: Diện tích, Giá tiền, Vị trí, và Kết cấu nhà.
"""

# Hàm tạo prompt cho từng căn nhà
def create_prompt(raw_text):
    messages = [{"role": "system", "content": system_prompt}]
    
    # Thêm Few-shot examples
    for fs in few_shot_examples:
        messages.append({"role": "user", "content": f"Mô tả thô:\n{fs[col_text]}"})
        messages.append({"role": "assistant", "content": f"Mô tả chuẩn:\n{fs[col_gt]}"})
        
    # Thêm text cần dự đoán
    messages.append({"role": "user", "content": f"Mô tả thô:\n{raw_text}\n\nHãy tạo 'Mô tả chuẩn' cho căn nhà này:"})
    return messages

print("Đã setup xong cấu trúc Prompt Few-shot!")

Đã setup xong cấu trúc Prompt Few-shot!


In [20]:
predictions = []
performance = []

print(f"Bắt đầu gọi API cho TOÀN BỘ {len(test_data)} mẫu...")
for i, item in enumerate(test_data):
    raw_text = item[col_text]
    row_id = item[col_id]
    # Lấy groundtruth nếu có, không có thì để rỗng
    groundtruth_text = item[col_gt] if pd.notna(item[col_gt]) else ""
    
    messages = create_prompt(raw_text)
    start_time = time.time()
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.3
        )
        latency = time.time() - start_time
        
        result_text = response.choices[0].message.content
        tokens_used = response.usage.total_tokens
        
        # Giá gpt-4o-mini tham khảo
        cost = (response.usage.prompt_tokens * 0.150 / 1000000) + (response.usage.completion_tokens * 0.600 / 1000000)
        
        predictions.append({
            "id": row_id, 
            "raw_text": raw_text, 
            "predicted_summary": result_text,
            "groundtruth": groundtruth_text
        })
        performance.append({
            "id": row_id, 
            "latency_seconds": round(latency, 2), 
            "total_tokens": tokens_used, 
            "estimated_cost_usd": cost
        })
        print(f"Đã xong mẫu {i+1}/{len(test_data)} (ID: {row_id}) - Thời gian: {round(latency, 2)}s")
    except Exception as e:
        print(f"Lỗi ở mẫu {i+1} (ID: {row_id}): {e}")

# Lưu kết quả ra file JSON
with open('predictions.json', 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=4)
    
with open('performance.json', 'w', encoding='utf-8') as f:
    json.dump(performance, f, ensure_ascii=False, indent=4)

print("\nHoàn thành! Đã lưu file predictions.json và performance.json cho 37 case.")

Bắt đầu gọi API cho TOÀN BỘ 37 mẫu...
Đã xong mẫu 1/37 (ID: SYS-20264727-938) - Thời gian: 10.02s
Đã xong mẫu 2/37 (ID: SYS-MP75438B-O8) - Thời gian: 4.89s
Đã xong mẫu 3/37 (ID: SYS-MP756313-W) - Thời gian: 5.49s
Đã xong mẫu 4/37 (ID: SYS-MP756P72-9B) - Thời gian: 5.68s
Đã xong mẫu 5/37 (ID: SYS-MP757WNV-DN) - Thời gian: 5.69s
Đã xong mẫu 6/37 (ID: SYS-MP758TA8-AN) - Thời gian: 5.14s
Đã xong mẫu 7/37 (ID: SYS-MP759ESN-94) - Thời gian: 6.83s
Đã xong mẫu 8/37 (ID: SYS-MP759ZIK-MM) - Thời gian: 4.91s
Đã xong mẫu 9/37 (ID: SYS-MP75AKU9-9R) - Thời gian: 339.99s
Đã xong mẫu 10/37 (ID: SYS-MP75ES0J-9K) - Thời gian: 6.27s
Đã xong mẫu 11/37 (ID: SYS-MP75GT4Y-GN) - Thời gian: 5.01s
Đã xong mẫu 12/37 (ID: SYS-MP75Z7G0-H3) - Thời gian: 7.18s
Đã xong mẫu 13/37 (ID: SYS-MP75ZR8R-C4) - Thời gian: 5.2s
Đã xong mẫu 14/37 (ID: SYS-MP760LYQ-AC) - Thời gian: 6.27s
Đã xong mẫu 15/37 (ID: SYS-MP760YOH-BW) - Thời gian: 7.36s
Đã xong mẫu 16/37 (ID: SYS-MP761FGI-DF) - Thời gian: 4.79s
Đã xong mẫu 17/37 (ID: SY

### Bước 4: Đánh giá Tự động từng Case (37 ID)
Tính ROUGE-L, BERTScore (chỉ những ID có Groundtruth mới có điểm, còn lại N/A), Specs Pres, Latency và Cost cho từng ID một và xuất ra DataFrame gồm 37 dòng.

In [21]:
#!pip install rouge_score bert_score evaluate
import json
import re
import numpy as np
import pandas as pd

try:
    from evaluate import load
except ImportError:
    print("Chưa cài đặt thư viện đánh giá. Hãy gỡ comment dòng !pip install...")

rouge = load('rouge')
bertscore = load('bertscore')

with open('predictions.json', 'r', encoding='utf-8') as f:
    preds_data = json.load(f)
    
with open('performance.json', 'r', encoding='utf-8') as f:
    perf_data = json.load(f)

def extract_numbers(text):
    return set(re.findall(r'\b\d+(?:\.\d+)?\b', text))

results_list = []

for pred, perf in zip(preds_data, perf_data):
    row_id = pred['id']
    raw_text = pred['raw_text']
    predicted_summary = pred['predicted_summary']
    groundtruth = pred['groundtruth']
    
    latency = perf['latency_seconds']
    cost_usd = perf['estimated_cost_usd']
    cost_vnd = cost_usd * 25400
    
    # TÍNH ROUGE-L & BERTScore (Nếu có groundtruth)
    if groundtruth and str(groundtruth).strip() != "":
        rouge_res = rouge.compute(predictions=[predicted_summary], references=[groundtruth])
        r_score = rouge_res['rougeL'] * 100
        
        bert_res = bertscore.compute(predictions=[predicted_summary], references=[groundtruth], lang='vi')
        b_score = bert_res['f1'][0] * 100
        
        r_score_str = f"{r_score:.2f}%"
        b_score_str = f"{b_score:.2f}%"
    else:
        r_score_str = "N/A"
        b_score_str = "N/A"
        
    # TÍNH SPECS PRESERVATION
    raw_nums = extract_numbers(raw_text)
    pred_nums = extract_numbers(predicted_summary)
    if len(raw_nums) == 0:
        specs_score = 100.0
    else:
        matched = raw_nums.intersection(pred_nums)
        specs_score = (len(matched) / len(raw_nums)) * 100
        
    # Tích hợp vào bảng
    results_list.append({
        "ID": row_id,
        "ROUGE-L": r_score_str,
        "BERTScore": b_score_str,
        "Specs Pres.": f"{specs_score:.2f}%",
        "Độ Trễ": f"{latency:.2f} s",
        "Chi Phí (VNĐ)": f"{int(cost_vnd):,} đ"
    })

df_all = pd.DataFrame(results_list)

# Lưu ra file CSV để dễ copy paste vào report
df_all.to_csv("evaluation_37_cases.csv", index=False, encoding='utf-8-sig')

print("Đã lưu kết quả 37 ID ra file evaluation_37_cases.csv. Dưới đây là dữ liệu bảng:")
display(df_all)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Đã lưu kết quả 37 ID ra file evaluation_37_cases.csv. Dưới đây là dữ liệu bảng:


,ID,ROUGE-L,BERTScore,Specs Pres.,Độ Trễ,Chi Phí (VNĐ)
0,SYS-20264727-938,37.53%,75.94%,44.44%,10.02 s,15 đ
1,SYS-MP75438B-O8,59.39%,83.08%,44.44%,4.89 s,13 đ
2,SYS-MP756313-W,45.79%,83.44%,61.54%,5.49 s,14 đ
3,SYS-MP756P72-9B,42.12%,77.62%,50.00%,5.68 s,14 đ
4,SYS-MP757WNV-DN,49.91%,81.88%,53.85%,5.69 s,13 đ
5,SYS-MP758TA8-AN,43.28%,83.06%,33.33%,5.14 s,14 đ
6,SYS-MP759ESN-94,50.00%,82.25%,50.00%,6.83 s,14 đ
7,SYS-MP759ZIK-MM,41.21%,78.29%,44.44%,4.91 s,13 đ
8,SYS-MP75AKU9-9R,38.40%,73.71%,52.94%,339.99 s,15 đ
9,SYS-MP75ES0J-9K,42.88%,82.22%,42.86%,6.27 s,14 đ
